# 02 · Injection-recovery: what *could* we have found?

Notebook 01 showed that the pipeline finds planets that are known to be there. That doesn't tell us what it **misses**. Without that, "I searched this star and found nothing" means very little.

**Injection-recovery** answers it with ground truth:
1. Take a real light curve of a star with no signal of its own.
2. Multiply in a fake planet transit with a known period and size.
3. Run the *exact same* search as for real data.
4. Did the fake planet come back? Repeat a few hundred times.

The result is a **sensitivity map** for this star and this pipeline.

In [ ]:
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import pandas as pd
import injection_recovery as ir
import transit_search as ts

%matplotlib inline

## 1. A quiet star

`quiet_stars.csv` was made with `ir.find_quiet_stars()`. It lists stars from the TESS Input Catalog with 2-minute data where the search, run *before* any injection, finds no candidate. We start with a near-solar twin.

*CDPP* (combined differential photometric precision) is the noise on a 1-hour average, in ppm. It's the standard "how quiet is this star" number.

In [ ]:
stars = pd.read_csv("../quiet_stars.csv")
star = stars.iloc[0]
display(stars[["target", "teff", "stellar_radius", "stellar_density", "cdpp_1h_ppm", "baseline_sde"]])

lc = ts.download_light_curve(star["target"], sectors=[int(star["sectors"])]).remove_nans().normalize()

## 2. Inject one planet by hand

A 2 Earth-radius planet on a 4.2-day orbit. Its depth follows from the radius ratio, and its duration from Kepler's third law and the star's density.

In [ ]:
period, radius_earth, impact = 4.2, 2.0, 0.3
depth = (radius_earth / (ir.EARTH_RADII_PER_SUN * star["stellar_radius"])) ** 2
duration = ir.transit_duration_days(period, star["stellar_density"], impact)
t0 = lc.time.value.min() + 1.3
print(f"depth = {depth * 1e6:.0f} ppm, duration = {duration * 24:.2f} h")

injected = ir.inject_transit(lc, period, t0, depth, duration, impact)

fig, ax = plt.subplots(figsize=(11, 3.5))
lc.plot(ax=ax, label="original", color="0.6")
injected.plot(ax=ax, label="with injected planet", color="k", lw=0.5);

You probably can't see the dips by eye. Now run the normal search:

In [ ]:
found = ts.search_light_curve(injected, name="injected 2 R_earth planet", show=True)
injection = {"injected_period": period, "injected_t0": t0, "injected_duration_hours": duration * 24}
print("status:", ir.classify(injection, found))

## 3. Hundreds of injections

`ir.run_injections()` draws random periods (0.5–13 days) and depths (30–5000 ppm), both uniform in log, with random impact parameters. It runs the full search on each, in parallel. Each injection is labelled:

| status | meaning |
|---|---|
| **recovered** | right period (±1%), right transit times, passed SDE ≥ 9 and SNR ≥ 7 |
| alias | found at 2× or ½× the period |
| other signal | a candidate was found, but at an unrelated period |
| below threshold | nothing passed the SDE/SNR cuts |

A full run (300 per star) takes ~15 minutes per star on 4 cores. Run `python run_injection_recovery.py` from the repo root, then load the results here. Or try a quick `n=40` below.

In [ ]:
from pathlib import Path

saved = Path("../results/injections") / f"{ts.safe_filename(star['target'])}.csv"
if saved.exists():
    results = pd.read_csv(saved)
else:
    results = ir.run_injections(lc, n=40, stellar_radius=star["stellar_radius"],
                                stellar_density=star["stellar_density"])
results["status"].value_counts()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.8))
ir.plot_recovery_map(results, title="Recovery fraction", ax=ax1)
ir.plot_recovery_vs_snr(results, title="Recovery vs. expected SNR", ax=ax2)
fig.tight_layout()

## 4. Reading the maps

- **Map**: recovery is high for big planets and short periods (more transits) and falls off for small planets at long periods. The edge of the dark region is this star's detection limit.
- **Recovery vs. expected SNR** should be a step: ~0 well below the SNR threshold, ~1 well above it. Two things to look for:
  - Where the step sits. If it's well right of 7, the search is losing SNR somewhere, for example to flattening.
  - Whether it reaches 1. Misses at high SNR are pipeline bugs, not noise. Look at those rows (`results.query("expected_snr > 30 and not recovered")`) and plot them.
- **Missed injections are the most useful rows.** This test already found two bugs that are fixed now:
  - BLS preferred unphysically long boxes at short periods. Fix: `MAX_DUTY_CYCLE`.
  - The second, masked pass could invent signals. Fix: pass 2 only measures, it doesn't search.

## Experiments

1. Look at the missed high-SNR injections. Why were they missed?
2. Set `window_days=0.5` (pass `search_kwargs={"window_days": 0.5}` to `run_injections`). Which part of the map gets worse? (Hint: long durations.)
3. Compare stars: `results/injections/recovery_vs_radius.png` shows why M dwarfs are good hunting grounds for small planets.